# 单卡GPU 进行 ChatGLM3-6B模型 LORA 高效微调
本 Cookbook 将带领开发者使用 `AdvertiseGen` 对 ChatGLM3-6B 数据集进行 lora微调，使其具备专业的广告生成能力。

**硬件需求**

显存：24GB及以上（推荐使用30系或A10等sm80架构以上的NVIDIA显卡进行尝试）内存：16GB
RAM: 2.9 /16 GB
GPU RAM: 15.5/16.0 GB

## 一、准备数据集
我们使用 AdvertiseGen 数据集来进行微调。从 [Google Drive](https://drive.google.com/file/d/13_vf0xRTQsyneRKdD1bZIr93vBGOczrk/view?usp=sharing) 或者 [Tsinghua Cloud](https://cloud.tsinghua.edu.cn/f/b3f119a008264b1cabd1/?dl=1) 下载处理好的 AdvertiseGen 数据集，将解压后的 AdvertiseGen 目录放到本目录的 `/data/` 下。

### 1.1 AdvertiseGen 数据格式转换

In [ ]:
import json           # json：标准库，用于 JSON 格式的序列化（dumps）和反序列化（loads）
from typing import Union  # Union：类型注解，表示参数可以是多种类型之一
from pathlib import Path  # Path：面向对象的文件系统路径类，支持路径拼接和文件操作


def _resolve_path(path: Union[str, Path]) -> Path:
    """
    将输入路径解析为绝对路径。

    参数：
        path (Union[str, Path]): 输入路径，支持字符串或 Path 对象，支持 '~' 用户目录展开

    返回值：
        Path: 展开用户目录并解析为绝对路径的 Path 对象
    """
    # expanduser() 将 '~' 展开为用户主目录，resolve() 将相对路径转为绝对路径
    return Path(path).expanduser().resolve()


def _mkdir(dir_name: Union[str, Path]):
    """
    递归创建目录，若目录已存在则跳过。

    参数：
        dir_name (Union[str, Path]): 要创建的目录路径，支持字符串或 Path 对象
    """
    # 将输入目录路径解析为绝对 Path 对象，类型：Path
    dir_name = _resolve_path(dir_name)
    if not dir_name.is_dir():  # 判断目录是否已存在，is_dir() 返回类型：bool
        # parents=True 表示递归创建中间目录，exist_ok=False 表示若目录已存在则报错（此处已提前判断）
        dir_name.mkdir(parents=True, exist_ok=False)


def convert_adgen(data_dir: Union[str, Path], save_dir: Union[str, Path]):
    """
    将 AdvertiseGen 原始数据集转换为 ChatGLM3 多轮对话微调格式（JSONL）。

    原始格式：每行为 {"content": "商品描述关键词", "summary": "广告文案"}
    目标格式：每行为 {"conversations": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}

    参数：
        data_dir (Union[str, Path]): 原始 AdvertiseGen 数据集目录，包含 train.json 和 dev.json
        save_dir (Union[str, Path]): 转换后数据的保存目录，会自动创建对应的子目录结构
    """
    def _convert(in_file: Path, out_file: Path):
        """
        将单个原始数据文件转换并保存为目标格式。

        参数：
            in_file (Path): 输入文件的绝对路径（原始 AdvertiseGen JSONL 文件）
            out_file (Path): 输出文件的绝对路径（转换后的对话格式 JSONL 文件）
        """
        # 确保输出文件的父目录存在，若不存在则递归创建
        _mkdir(out_file.parent)
        with open(in_file, encoding='utf-8') as fin:          # 以 UTF-8 编码打开原始输入文件
            with open(out_file, 'wt', encoding='utf-8') as fout:  # 以文本写模式打开输出文件
                for line in fin:  # 逐行读取原始文件，每行为一条 JSON 记录，类型：str
                    dct = json.loads(line)  # 将 JSON 字符串解析为 Python 字典，类型：dict
                    # 构建 ChatGLM3 对话格式的样本字典
                    # 'content' 作为用户输入（商品关键词），'summary' 作为 assistant 回复（广告文案）
                    sample = {'conversations': [
                        {'role': 'user', 'content': dct['content']},        # 用户消息（商品描述关键词）
                        {'role': 'assistant', 'content': dct['summary']}    # 助手回复（广告文案）
                    ]}
                    # 将样本字典序列化为 JSON 字符串并写入文件
                    # ensure_ascii=False 保留中文字符（不转义为 \uXXXX），类型写入：str + '\n'
                    fout.write(json.dumps(sample, ensure_ascii=False) + '\n')

    # 将数据目录解析为绝对 Path 对象，类型：Path
    data_dir = _resolve_path(data_dir)
    # 将保存目录解析为绝对 Path 对象，类型：Path
    save_dir = _resolve_path(save_dir)

    train_file = data_dir / 'train.json'  # 拼接训练集文件路径，类型：Path
    if train_file.is_file():  # 检查训练集文件是否存在，is_file() 返回类型：bool
        # 计算相对于 data_dir 的相对路径，并拼接到 save_dir 下，保持目录结构一致
        out_file = save_dir / train_file.relative_to(data_dir)  # 输出文件路径，类型：Path
        _convert(train_file, out_file)  # 转换训练集文件

    dev_file = data_dir / 'dev.json'  # 拼接验证集文件路径，类型：Path
    if dev_file.is_file():  # 检查验证集文件是否存在
        # 计算相对路径并拼接到保存目录，类型：Path
        out_file = save_dir / dev_file.relative_to(data_dir)
        _convert(dev_file, out_file)  # 转换验证集文件


# 执行数据转换：将原始 AdvertiseGen 目录的数据转换后保存到 AdvertiseGen_fix 目录
convert_adgen('data/AdvertiseGen', 'data/AdvertiseGen_fix')

## 二、使用命令行开始微调
接着，我们仅需要将配置好的参数以命令行的形式传参给程序，就可以使用命令行进行高效微调。

### 2.1 使用 LoRA 方法启动微调训练

In [ ]:
import os  # 导入标准库 os，用于在 Python 中设置环境变量（Windows 不支持 bash 内联变量赋值）

# ── 环境变量配置（Windows 下必须通过 os.environ 设置，不能使用 bash 的 KEY=VAL 前缀语法）──
os.environ['CUDA_VISIBLE_DEVICES'] = '0'      # 指定只使用第 0 号 GPU 进行训练
os.environ['NCCL_P2P_DISABLE'] = '1'          # 禁用 NCCL 点对点（P2P）通信，避免部分 GPU 架构的兼容问题
os.environ['NCCL_IB_DISABLE'] = '1'           # 禁用 NCCL InfiniBand 网络通信（无 IB 网卡的环境下必须禁用）

# ── 本地 ChatGLM3-6B 模型路径（HuggingFace Hub 缓存的快照目录）──
# 该路径由 HuggingFace 自动下载并缓存在 model/ChatGLM3/model/ 下，
# snapshots/<hash>/ 即为包含 config.json、pytorch_model 等完整权重文件的目录
MODEL_DIR = "model/ChatGLM3/model/models--THUDM--chatglm3-6b/snapshots/e9e0406d062cdb887444fe5bd546833920abd4ac"

# 启动单卡 LoRA 微调训练，各参数说明如下：
# finetune_hf.py    ：微调训练入口脚本（支持全参数微调、LoRA、P-Tuning v2）
# data/AdvertiseGen_fix ：训练数据目录，包含已转换为 ChatGLM3 对话格式的 JSON 文件（由 Cell 3 生成）
# MODEL_DIR         ：ChatGLM3-6B 预训练基础模型本地快照路径
# configs/lora.yaml ：LoRA 微调超参数配置文件（rank=8、alpha=32、max_steps=3000 等）
# -u：强制 Python 以无缓冲（unbuffered）模式运行，使训练日志和 tqdm 进度条实时刷新到 Jupyter 输出格
# 若不加 -u，Python 子进程默认使用块缓冲（~8KB），导致所有输出堆积后才一次性显示
!python -u finetune_hf.py data/AdvertiseGen_fix {MODEL_DIR} configs/lora.yaml

## 三、使用微调后的模型进行推理
在完成微调任务之后，我们可以查看到 `output` 文件夹下多了很多个`checkpoint-*`的文件夹，这些文件夹代表了训练的轮数。
我们选择最后一轮的微调权重，并使用inference进行导入。

### 3.1 加载微调检查点进行推理

In [ ]:
import os  # 导入标准库 os，用于设置环境变量

# ── 环境变量配置（Windows 下通过 os.environ 设置，与 Cell 6 保持一致）──
os.environ['CUDA_VISIBLE_DEVICES'] = '0'      # 指定只使用第 0 号 GPU 进行推理
os.environ['NCCL_P2P_DISABLE'] = '1'          # 禁用 NCCL P2P 通信
os.environ['NCCL_IB_DISABLE'] = '1'           # 禁用 NCCL InfiniBand 通信

# ── 检查点路径说明 ──
# configs/lora.yaml 中配置：max_steps=3000，save_steps=500
# 因此会生成检查点：checkpoint-500、1000、1500、2000、2500、3000
# 这里使用最终检查点 checkpoint-3000（原为 checkpoint-4000，但该步数不会被生成）
CHECKPOINT = "output/checkpoint-3000"  # 最终训练步数保存的 LoRA adapter 检查点目录，类型：str

# 使用微调后的 LoRA 检查点进行单轮对话推理，各参数说明如下：
# inference_hf.py   ：推理脚本，自动检测目录中是否含 adapter_config.json，
#                     若存在则以 PEFT 模式加载 LoRA adapter，否则加载标准模型
# CHECKPOINT        ：微调训练最终步保存的检查点目录，含 LoRA adapter 权重
# --prompt          ：输入提示文本，格式为 "属性#值*属性#值" 的商品关键词组合，
#                     模型将据此生成对应的广告文案
!python inference_hf.py {CHECKPOINT} --prompt "类型#裙*版型#显瘦*材质#网纱*风格#性感*裙型#百褶*裙下摆#压褶*裙长#连衣裙*裙衣门襟#拉链*裙衣门襟#套头*裙款式#拼接*裙款式#拉链*裙款式#木耳边*裙款式#抽褶*裙款式#不规则"

## 四、总结
到此位置，我们就完成了使用单张 GPU Lora 来微调 ChatGLM3-6B 模型，使其能生产出更好的广告。
在本章节中，你将会学会：
+ 如何使用模型进行 Lora 微调
+ 微调数据集的准备和对齐
+ 使用微调的模型进行推理